# 1일차 4교시 · 회귀 모델 평가 지표

**MAE, MSE, RMSE, R²**를 손 계산 예제와 당뇨 진행도 데이터로 확인합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             mean_absolute_percentage_error)

## 1. 이상치 하나가 지표를 얼마나 바꾸나

잔차(실제 − 예측) 5개. A는 큰 오차가 1건, B는 고르게 틀린 경우입니다.

In [ ]:
cases = {"A: 큰 오차 1건": np.array([2, -1, 3, -2, 8]),
         "B: 고르게 틀림": np.array([2, -1, 3, -2, 2])}
rows = []
for name, residual in cases.items():
    mae = np.mean(np.abs(residual))
    mse = np.mean(residual ** 2)
    rows.append({"경우": name, "MAE": mae, "MSE": mse, "RMSE": round(np.sqrt(mse), 2)})
pd.DataFrame(rows)

큰 오차 하나에 MSE가 훨씬 크게 반응합니다. **RMSE가 MAE보다 훨씬 크면 가끔 크게 틀린다는 신호**입니다.

## 2. 당뇨 진행도 예측

442명의 환자 정보 10개로 1년 뒤 질병 진행도(수치)를 예측합니다.

In [ ]:
X, y = load_diabetes(return_X_y=True)
print("데이터 크기:", X.shape)
print(f"목표값 범위: {y.min():.0f} ~ {y.max():.0f}, 평균 {y.mean():.1f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
reg = LinearRegression().fit(X_train, y_train)
y_pred = reg.predict(X_test)

In [ ]:
def regression_report(y_true, y_pred):
    return {"MAE": round(mean_absolute_error(y_true, y_pred), 1),
            "RMSE": round(mean_squared_error(y_true, y_pred) ** 0.5, 1),
            "R2": round(r2_score(y_true, y_pred), 3)}

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
pd.DataFrame([{"모델": "기준선 (평균으로 예측)", **regression_report(y_test, baseline.predict(X_test))},
              {"모델": "선형 회귀", **regression_report(y_test, y_pred)}])

**업무 언어로 바꿔 보기**

- "진행도(25~346, 평균 152)를 예측할 때 평균적으로 약 41 정도 틀립니다."
- 기준선(평균으로만 예측)은 R²가 0 근처입니다. 선형 회귀는 그보다 낫지만 변동의 절반 정도만 설명합니다.

## 3. 잔차 그림

그래프의 글자는 Colab에서 한글 폰트가 깨지지 않도록 영어로 적었습니다.

In [ ]:
residual = y_test - y_pred
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residual, alpha=0.6)
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Predicted value")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual plot")
plt.show()

0 주변에 고르게 흩어져 있으면 좋습니다. 곡선 모양이나 깔때기 모양이 보이면 모델이 놓친 패턴이 있다는 뜻입니다.

## 4. [선택] MAPE와 그 함정

In [ ]:
print("당뇨 데이터 MAPE:", round(mean_absolute_percentage_error(y_test, y_pred), 3))

# 실제값이 0에 가까운 경우가 섞이면
y_true_small = np.array([100, 120, 90, 1, 110])
y_pred_small = np.array([95, 125, 92, 6, 108])
print("실제값 1이 섞인 경우 MAPE:", round(mean_absolute_percentage_error(y_true_small, y_pred_small), 3))

실제값 1을 6으로 예측한 한 건(오차 5) 때문에 MAPE가 1(=100%)을 넘습니다. 실제값이 0 근처인 데이터에는 MAPE를 쓰지 않습니다.

## 5. 연습 문제

잔차가 `[1, -1, 1, -1, 10]`일 때 MAE, MSE, RMSE를 계산해 보세요.

In [ ]:
residual = np.array([1, -1, 1, -1, 10])
# 여기에 계산 코드를 작성하세요